# 🩺 Pharmacovigilance Signal Triage Copilot — Agent AI

## Scop
Acest notebook implementează agentul AI de triaj al semnalelor de siguranță farmaceutică.
Agentul primește instrucțiuni în limbaj natural și execută autonom un flux complet de investigație:

1. **Filtrare temporală** — selectează rapoartele FAERS din intervalul cerut
2. **Normalizare medicamente** — convertește denumiri comerciale la INN via RxNorm API
3. **Normalizare reacții** — standardizează termenii la nomenclatura MedDRA
4. **Calcul metrici** — PRR, ROR și analiză de trend lunar (criteriul Evans 2001)
5. **Drill-down** — extrage rapoarte individuale reprezentative
6. **Signal Packet PDF** — generează automat un document exportabil cu grafice

## Stack tehnic
- **LLM:** Llama 3.3 70B (open-source Meta, inferență via Groq API)
- **Agent framework:** LangGraph `create_react_agent` — pattern ReAct (Reason + Act)
- **Tools:** 6 funcții Python decorate cu `@tool` din `langchain_core`
- **UI:** Gradio — interfață web cu link public pentru demonstrație
- **Date:** FDA FAERS (Adverse Event Reporting System) — date publice, 2022–2024

## 0. Instalare Dependențe

Rulați o singură dată per sesiune. Instalăm toate pachetele necesare într-o singură comandă.

In [33]:
%pip install langchain langchain-core langchain-groq \
             langgraph python-dotenv pandas fpdf requests matplotlib seaborn gradio -q


## 1. Configurare Chei API

Cheile API nu se hardcodează niciodată în cod — se citesc dintr-un fișier `.env` local,
care este adăugat în `.gitignore` și nu ajunge pe Git.

Creați fișierul `.env` în același director cu notebook-ul:
```
GROQ_API_KEY=cheia_voastra_de_la_console.groq.com
FDA_API_KEY=cheia_voastra_de_la_open.fda.gov
```

In [34]:
import os
from dotenv import load_dotenv

load_dotenv()
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
FDA_API_KEY  = os.getenv('FDA_API_KEY')

if not GROQ_API_KEY:
    raise EnvironmentError('GROQ_API_KEY nu a fost gasita in .env!')

print('Chei API incarcate cu succes.')
print(f'  GROQ: {"OK" if GROQ_API_KEY else "LIPSA"}')
print(f'  FDA:  {"OK" if FDA_API_KEY else "LIPSA (optional)"}')

Chei API incarcate cu succes.
  GROQ: OK
  FDA:  OK


## 2. Importuri și Încărcare Date

Citim CSV-ul generat de Notebook-ul 01 (EDA), care conține rapoartele FAERS
prelucrate pentru anii 2022–2024.

Aplicăm **deduplicarea automată** pe combinația `(Report_ID, Drug_Name, Reaction)` —
un același caz poate apărea de mai multe ori în FAERS dacă e raportat de mai mulți
expeditori (pacient, medic, producător). Fără deduplicare, metricile PRR/ROR ar fi
supraevaluate.

In [36]:
import pandas as pd
import numpy as np
import requests
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time, re, os
from datetime import datetime

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

CSV_PATH = '../data/cleaned_faers_data.csv'

# Fallback: descarcam date demo daca CSV-ul nu exista
if not os.path.exists(CSV_PATH):
    print('CSV nu gasit. Descarcam esantion demo de la FDA...')
    os.makedirs('../data', exist_ok=True)
    resp = requests.get(
        'https://api.fda.gov/drug/event.json',
        params={'search': 'receivedate:[20230101+TO+20241231]', 'limit': 1000},
        headers={'User-Agent': 'PV-Copilot/1.0'}
    )
    if resp.status_code == 200:
        raw = resp.json().get('results', [])
        rows = []
        for r in raw:
            rid  = r.get('safetyreportid', 'N/A')
            date = r.get('receivedate', '20230101')
            sc   = r.get('patient', {}).get('patientsex', '0')
            sex  = 'M' if sc == '1' else ('F' if sc == '2' else 'Unknown')
            rxns = [rx.get('reactionmeddrapt','').upper()
                    for rx in r.get('patient',{}).get('reaction',[])]
            for drug in r.get('patient',{}).get('drug',[]):
                dname = drug.get('medicinalproduct','').upper()
                role  = 'Suspect' if drug.get('drugcharacterization','0') == '1' else 'Concomitent'
                for rxn in rxns:
                    if dname and rxn:
                        rows.append({'Report_ID': rid, 'Date': date, 'Sex': sex,
                                     'Drug_Name': dname, 'Role': role, 'Reaction': rxn})
        pd.DataFrame(rows).to_csv(CSV_PATH, index=False)
        print(f'Esantion salvat: {len(rows)} inregistrari')
    else:
        raise RuntimeError(f'Eroare FDA API: {resp.status_code}')

date_brute = pd.read_csv(CSV_PATH)
date_brute['An_Raport'] = (
    date_brute['Date'].astype(str).str[:4]
    .apply(lambda x: int(x) if x.isdigit() else 0)
)

# Deduplicare automata
date_unice = date_brute.drop_duplicates(subset=['Report_ID', 'Drug_Name', 'Reaction']).copy()

# FIX: salvam date_unice deduplicat ca CSV separat pentru reproductibilitate
# Daca CSV-ul deja exista cu deduplicare, nu il rescriem inutil
DEDUP_PATH = '../data/cleaned_faers_data_deduped.csv'
if not os.path.exists(DEDUP_PATH):
    date_unice.to_csv(DEDUP_PATH, index=False)
    print(f'Date deduplicate salvate: {DEDUP_PATH}')

print(f'Date incarcate: {len(date_brute):,} brute => {len(date_unice):,} dupa deduplicare')
print(f'Medicamente unice: {date_unice["Drug_Name"].nunique():,}')
print(f'Reactii unice:     {date_unice["Reaction"].nunique():,}')
date_unice.head(3)


Date incarcate: 162,798 brute => 101,919 dupa deduplicare
Medicamente unice: 2,812
Reactii unice:     2,694


,Report_ID,Date,Sex,Drug_Name,Role,Reaction,An_Raport
0,20268558,20220101,F,OXYCONTIN,Suspect,DRUG DEPENDENCE,2022
1,20268558,20220101,F,OXYCONTIN,Suspect,PAIN,2022
2,20268558,20220101,F,OXYCONTIN,Suspect,EMOTIONAL DISTRESS,2022


## 3. Definirea Tool-urilor Agentului

Fiecare funcție de mai jos este un **tool** pe care agentul îl poate apela autonom.
Decoratorul `@tool` din `langchain_core.tools` face două lucruri:
1. Expune funcția agentului ca acțiune disponibilă
2. Folosește docstring-ul funcției ca descriere — agentul citește descrierea ca să știe
   când și cum să apeleze tool-ul

Variabila globală `_df_curent` este starea partajată între tool-uri — fiecare tool
citește sau modifică același dataframe filtrat activ.

### 3.1 Tool: Filtrare Temporala

In [4]:
from langchain_core.tools import tool

_df_curent = None  # va fi initializat inainte de fiecare rulare a agentului

@tool
def filtreaza_dupa_timp(an_start: int, an_stop: int) -> str:
    """Filtreaza rapoartele FAERS pe intervalul de ani [an_start, an_stop].
    Parametri:
      - an_start: primul an al intervalului (ex: 2023)
      - an_stop: ultimul an al intervalului inclusiv (ex: 2024)
    """
    global _df_curent
    inainte = len(_df_curent) if _df_curent is not None else 0
    _df_curent = date_unice[
        (date_unice['An_Raport'] >= an_start) & (date_unice['An_Raport'] <= an_stop)
    ].copy()
    dupa = len(_df_curent)
    return f'Filtrare OK: {inainte:,} => {dupa:,} rapoarte pentru {an_start}-{an_stop}.'

print('Tool filtreaza_dupa_timp inregistrat.')


Tool filtreaza_dupa_timp inregistrat.


### 3.2 Tool: Normalizare Medicament (RxNorm API)

In [5]:
@tool
def normalizeaza_medicament_rxnorm(nume_comercial: str) -> str:
    """Cauta denumirea generica INN a unui medicament folosind API-ul RxNorm (gratuit, public).
    Parametri:
      - nume_comercial: denumirea comerciala/brand a medicamentului (ex: DUPIXENT)
    """
    try:
        url = f'https://rxnav.nlm.nih.gov/REST/rxcui.json?name={nume_comercial}&search=2'
        r = requests.get(url, timeout=8)
        if r.status_code != 200:
            return f'RxNorm indisponibil (HTTP {r.status_code}). Se pastreaza: {nume_comercial.upper()}.'
        data = r.json()
        ids  = data.get('idGroup', {}).get('rxnormId', [])
        if not ids:
            return f'Nu s-a gasit RxCUI pentru {nume_comercial}. Se pastreaza original.'
        rxcui = ids[0]
        url2 = f'https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/allProperties.json?prop=names'
        r2   = requests.get(url2, timeout=8)
        if r2.status_code == 200:
            props = r2.json().get('propConceptGroup', {}).get('propConcept', [])
            inn_l = [p['propValue'] for p in props if 'INN' in p.get('propName','').upper()]
            inn   = inn_l[0] if inn_l else data.get('idGroup',{}).get('name', nume_comercial).upper()
            return f'Normalizare RxNorm: {nume_comercial} => INN: {inn} (RxCUI: {rxcui})'
        return f'RxCUI gasit: {rxcui} pentru {nume_comercial}.'
    except requests.exceptions.Timeout:
        return f'Timeout RxNorm. Se pastreaza: {nume_comercial.upper()}.'
    except Exception as e:
        return f'Eroare RxNorm: {e}. Se pastreaza: {nume_comercial.upper()}.'

print('Tool normalizeaza_medicament_rxnorm inregistrat.')


Tool normalizeaza_medicament_rxnorm inregistrat.


### 3.3 Tool: Normalizare Reactie (MedDRA)

> **⚠️ Limitare cunoscuta:** MedDRA (Medical Dictionary for Regulatory Activities) este un dicționar cu **licență comercială plătită** — accesul programatic complet necesită abonament organizațional. UMLS API (NLM/NIH) oferă o punte gratuită, dar necesită înregistrare individuală.
>
> **Soluție adoptată în prototip:** mapping local pe cei mai frecvenți ~30 termeni din FAERS (sinonime, greșeli de ortografie, abrevieri) la Preferred Term MedDRA. Termenii care nu se regăsesc în mapping sunt returnați uppercase — aceasta este o limitare documentată și nu o eroare de implementare.
>
> În producție, tool-ul ar apela `https://uts-ws.nlm.nih.gov/rest/search/` cu autentificare UMLS.


In [6]:
@tool
def normalizeaza_reactie_meddra(nume_reactie: str) -> str:
    """Standardizeaza un termen de reactie adversa la nivelul Preferred Term MedDRA.
    Foloseste un mapping local pentru termenii frecventi din FAERS.
    In productie: UMLS API (https://uts-ws.nlm.nih.gov) cu autentificare.
    Parametri:
      - nume_reactie: termenul de reactie de normalizat
    """
    # Mapping local: sinonime/abrevieri frecvente => MedDRA Preferred Term
    MEDDRA_MAP = {
        # Reactii oculare
        'DRY EYES': 'DRY EYE',
        'DRY EYE DISEASE': 'DRY EYE',
        'OCULAR DRYNESS': 'DRY EYE',
        'EYE DRYNESS': 'DRY EYE',
        # Infectii
        'INFECTION': 'INFECTION',
        'INFECTIONS': 'INFECTION',
        'BACTERIAL INFECTION': 'BACTERIAL INFECTION',
        # Tractul GI
        'NAUSEA': 'NAUSEA',
        'NAUSEA/VOMITING': 'NAUSEA',
        'FEELING SICK': 'NAUSEA',
        'VOMITING': 'VOMITING',
        'VOMIT': 'VOMITING',
        'DIARRHOEA': 'DIARRHOEA',
        'DIARRHEA': 'DIARRHOEA',  # US -> MedDRA (UK spelling)
        # Neurologice
        'HEADACHE': 'HEADACHE',
        'HEAD PAIN': 'HEADACHE',
        'MIGRAINE': 'MIGRAINE',
        'DIZZINESS': 'DIZZINESS',
        'FATIGUE': 'FATIGUE',
        'TIREDNESS': 'FATIGUE',
        # Cutanate
        'ECZEMA': 'ECZEMA',
        'DERMATITIS': 'DERMATITIS',
        'ATOPIC DERMATITIS': 'DERMATITIS ATOPIC',
        'RASH': 'RASH',
        'SKIN RASH': 'RASH',
        # Pulmonare
        'PNEUMONITIS': 'PNEUMONITIS',
        'LUNG INFLAMMATION': 'PNEUMONITIS',
        'PNEUMONIA': 'PNEUMONIA',
        'DYSPNOEA': 'DYSPNOEA',
        'DYSPNEA': 'DYSPNOEA',
        'SHORTNESS OF BREATH': 'DYSPNOEA',
        # Durere
        'PAIN': 'PAIN',
        'ARTHRALGIA': 'ARTHRALGIA',
        'JOINT PAIN': 'ARTHRALGIA',
        'MYALGIA': 'MYALGIA',
        'MUSCLE PAIN': 'MYALGIA',
    }
    termen = ' '.join(nume_reactie.strip().upper().split())
    pt = MEDDRA_MAP.get(termen, termen)  # fallback la original daca nu e in map
    sursa = 'mapping local' if termen in MEDDRA_MAP else 'termen original (nemapt)'
    return (
        f'Termen MedDRA standardizat: PT = "{pt}" '
        f'(sursa: {sursa}; in productie: UMLS API cu autentificare)'
    )

print('Tool normalizeaza_reactie_meddra inregistrat.')


Tool normalizeaza_reactie_meddra inregistrat.


### 3.4 Tool: Calculul Metricilor PRR, ROR si Trend

In [7]:
@tool
def calculeaza_metrici_si_trend(nume_medicament: str, reactie_adversa: str) -> str:
    """Calculeaza PRR, ROR si analiza trend pentru o pereche medicament-reactie.
    Criteriu Evans 2001: semnal pozitiv daca PRR >= 2 si minim 3 cazuri.
    Parametri:
      - nume_medicament: medicamentul analizat
      - reactie_adversa: reactia adversa analizata
    """
    df = _df_curent
    N  = len(df) if df is not None else 0
    if N == 0:
        return 'Nu exista date. Ruleaza mai intai filtreaza_dupa_timp.'

    mask_d = df['Drug_Name'].str.contains(nume_medicament, case=False, na=False)
    mask_r = df['Reaction'].str.contains(reactie_adversa,  case=False, na=False)

    a = int(len(df[ mask_d &  mask_r]))
    b = int(len(df[ mask_d & ~mask_r]))
    c = int(len(df[~mask_d &  mask_r]))
    d = int(len(df[~mask_d & ~mask_r]))

    if a < 3:
        return (f'Prea putine cazuri (a={a}) pentru {nume_medicament} + {reactie_adversa}. '
                f'Criteriul Evans (min 3 cazuri) nu e indeplinit.')

    prr = (a / (a+b+1e-9)) / ((c / (c+d+1e-9)) + 1e-9)
    ror = (a*d) / ((b*c) + 1e-9) if (b > 0 and c > 0 and d > 0) else 0.0
    interp = 'SEMNAL POZITIV (PRR >= 2)' if prr >= 2 else 'Sub pragul de semnal (PRR < 2)'

    # Trend pe luni (ultimele 3 luni vs restul)
    df_s = df[mask_d & mask_r].copy()
    df_s['Luna'] = pd.to_datetime(df_s['Date'].astype(str), format='%Y%m%d', errors='coerce').dt.to_period('M')
    luna_max      = df_s['Luna'].max()
    luna_baseline = luna_max - 3

    recent   = int(len(df_s[df_s['Luna'] >= luna_baseline]))
    baseline = int(len(df_s[df_s['Luna'] <  luna_baseline]))
    factor   = recent / (baseline + 1)

    if factor > 2.0:
        trend = f'EMERGENT (crestere {factor:.1f}x in ultimele 3 luni vs baseline)'
    elif factor > 1.2:
        trend = f'IN CRESTERE (+{(factor-1)*100:.0f}% in ultimele 3 luni vs baseline)'
    else:
        trend = f'Stabil (factor {factor:.2f})'

    scor = min(10.0, (min(prr, 10)/10)*6 + (min(factor, 5)/5)*4)

    return (
        f'METRICI DISPROPORTIONALITATE\n'
        f'  Tabel 2x2: a={a}, b={b}, c={c}, d={d}, N={N}\n'
        f'  PRR={prr:.3f} => {interp}\n'
        f'  ROR={ror:.3f}\n'
        f'  Trend (ultimele 3 luni vs baseline): {recent} vs {baseline} cazuri => {trend}\n'
        f'  Scor Prioritizare: {scor:.2f}/10.00\n'
        f'  PRR_VALUE={prr:.4f}|ROR_VALUE={ror:.4f}|SCOR={scor:.4f}|A={a}|TREND={factor:.4f}'
    )

print('Tool calculeaza_metrici_si_trend inregistrat.')

Tool calculeaza_metrici_si_trend inregistrat.


### 3.5 Tool: Drill-Down Rapoarte Individuale

In [8]:
@tool
def extrage_cazuri_reprezentative(nume_medicament: str, reactie_adversa: str) -> str:
    """Extrage primele 5 rapoarte individuale pentru perechea medicament-reactie (drill-down).
    Parametri:
      - nume_medicament: medicamentul investigat
      - reactie_adversa: reactia adversa investigata
    """
    cazuri = _df_curent[
        _df_curent['Drug_Name'].str.contains(nume_medicament, case=False, na=False) &
        _df_curent['Reaction'].str.contains(reactie_adversa, case=False, na=False)
    ].head(5)

    if cazuri.empty:
        return f'Nu s-au gasit rapoarte pentru {nume_medicament} + {reactie_adversa}.'

    linii = [f'{len(cazuri)} cazuri reprezentative:']
    for i, (_, r) in enumerate(cazuri.iterrows(), 1):
        linii.append(
            f'  [{i}] ID:{r["Report_ID"]} | Data:{r["Date"]} | '
            f'Sex:{r.get("Sex","N/A")} | Rol:{r["Role"]}'
        )
    return '\n'.join(linii)

print('Tool extrage_cazuri_reprezentative inregistrat.')


Tool extrage_cazuri_reprezentative inregistrat.


### 3.6 Tool: Generare Signal Packet PDF cu Grafice

In [27]:
from fpdf import FPDF

@tool
def genereaza_pachet_pdf(
    medicament: str,
    reactie: str,
    metrici_text: str,
    recomandari_ai: str,
    exemple_rapoarte: str
) -> str:
    """Genereaza Signal Packet PDF complet cu grafice PRR/ROR, trend lunar, drill-down si recomandari AI.
    Parametri:
      - medicament: medicamentul investigat
      - reactie: reactia adversa
      - metrici_text: output de la calculeaza_metrici_si_trend
      - recomandari_ai: recomandari formulate de agent
      - exemple_rapoarte: output de la extrage_cazuri_reprezentative
    """
    def exf(pattern, text, default=0.0):
        m = re.search(pattern, text)
        return float(m.group(1)) if m else default

    def ss(s):
        replacements = {
            'ș': 's', 'ț': 't', 'ă': 'a', 'î': 'i', 'â': 'a',
            'Ș': 'S', 'Ț': 'T', 'Ă': 'A', 'Î': 'I', 'Â': 'A'
        }
        for char, rep in replacements.items():
            s = s.replace(char, rep)
        return s.encode('latin-1', errors='replace').decode('latin-1')

    # Extragem valorile din metrici_text daca exista
    prr_v  = exf(r'PRR_VALUE=([\d\.]+)', metrici_text)
    ror_v  = exf(r'ROR_VALUE=([\d\.]+)', metrici_text)
    scor_v = exf(r'SCOR=([\d\.]+)', metrici_text)
    a_m    = re.search(r'A=(\d+)', metrici_text)
    a_v    = int(a_m.group(1)) if a_m else 0

    # FALLBACK: daca Llama nu a pasat valorile, le recalculam direct din date
    if a_v == 0 and _df_curent is not None and len(_df_curent) > 0:
        df = _df_curent
        N  = len(df)
        mask_d = df['Drug_Name'].str.contains(medicament, case=False, na=False)
        mask_r = df['Reaction'].str.contains(reactie,    case=False, na=False)
        a_v = int(len(df[ mask_d &  mask_r]))
        b   = int(len(df[ mask_d & ~mask_r]))
        c   = int(len(df[~mask_d &  mask_r]))
        d   = int(len(df[~mask_d & ~mask_r]))
        if a_v >= 3:
            prr_v  = (a_v / (a_v+b+1e-9)) / ((c / (c+d+1e-9)) + 1e-9)
            ror_v  = (a_v*d) / ((b*c) + 1e-9) if (b > 0 and c > 0 and d > 0) else 0.0
            factor = a_v / (max(b, 1))
            scor_v = min(10.0, (min(prr_v, 10)/10)*6 + (min(factor, 5)/5)*4)

    # FALLBACK pentru exemple_rapoarte
    if not exemple_rapoarte.strip() or 'pentru' in exemple_rapoarte.lower() and len(exemple_rapoarte) < 60:
        if _df_curent is not None:
            cazuri = _df_curent[
                _df_curent['Drug_Name'].str.contains(medicament, case=False, na=False) &
                _df_curent['Reaction'].str.contains(reactie, case=False, na=False)
            ].head(5)
            if not cazuri.empty:
                linii = [f'{len(cazuri)} cazuri reprezentative:']
                for i, (_, r) in enumerate(cazuri.iterrows(), 1):
                    linii.append(
                        f'  [{i}] ID:{r["Report_ID"]} | Data:{r["Date"]} | '
                        f'Sex:{r.get("Sex","N/A")} | Rol:{r["Role"]}'
                    )
                exemple_rapoarte = '\n'.join(linii)
            else:
                exemple_rapoarte = f'Nu s-au gasit cazuri pentru {medicament} + {reactie}.'

    # Grafice
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    cul = ['#e74c3c' if prr_v >= 2 else '#2ecc71', '#3498db', '#bdc3c7']
    axes[0].bar(['PRR', 'ROR', 'Prag (PRR=2)'], [prr_v, ror_v, 2.0], color=cul)
    axes[0].axhline(2.0, color='red', linestyle='--', linewidth=1.2, label='Prag Evans')
    axes[0].set_title(f'Disproportionalitate: {medicament[:12]}/{reactie[:12]}', fontsize=9, fontweight='bold')
    axes[0].set_ylabel('Valoare')
    axes[0].legend(fontsize=7)

    if _df_curent is not None:
        df_t = _df_curent[
            _df_curent['Drug_Name'].str.contains(medicament, case=False, na=False) &
            _df_curent['Reaction'].str.contains(reactie, case=False, na=False)
        ].copy()

        if not df_t.empty:
            df_t['Luna'] = pd.to_datetime(
                df_t['Date'].astype(str), format='%Y%m%d', errors='coerce'
            ).dt.to_period('M')
            td = df_t.groupby('Luna').size().reset_index(name='Cazuri')
            td['Luna_str'] = td['Luna'].astype(str)

            if len(td) > 1:
                axes[1].plot(range(len(td)), td['Cazuri'],
                             marker='o', color='#e74c3c', linewidth=2, markersize=5)
                axes[1].fill_between(range(len(td)), td['Cazuri'],
                                     alpha=0.25, color='#e74c3c')
                axes[1].set_xticks(range(len(td)))
                axes[1].set_xticklabels(td['Luna_str'], rotation=45, ha='right', fontsize=7)
                axes[1].set_title('Trend Cazuri pe Luna', fontsize=9, fontweight='bold')
                axes[1].set_ylabel('Nr. rapoarte')
            else:
                luna = td['Luna_str'].iloc[0] if len(td) > 0 else 'N/A'
                axes[1].text(0.5, 0.5,
                             f'Toate cazurile ({a_v})\ndintr-o singura luna ({luna})',
                             ha='center', va='center', transform=axes[1].transAxes, fontsize=9)
                axes[1].set_title('Trend Cazuri pe Luna', fontsize=9, fontweight='bold')
        else:
            axes[1].text(0.5, 0.5, 'Nu exista cazuri', ha='center', va='center',
                         transform=axes[1].transAxes)

    plt.tight_layout()
    safe  = re.sub(r'[^a-zA-Z0-9]', '_', medicament)[:15]
    gpath = f'/tmp/grafic_{safe}.png'
    plt.savefig(gpath, dpi=150, bbox_inches='tight')
    plt.close()

    # PDF
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    pdf.set_font('Arial', 'B', 15)
    pdf.cell(0, 10, 'PHARMACOVIGILANCE SIGNAL PACKET', ln=True, align='C')
    pdf.set_font('Arial', 'I', 9)
    pdf.cell(0, 6, f'Generat automat | {datetime.now().strftime("%d.%m.%Y %H:%M")}', ln=True, align='C')
    pdf.ln(3)
    pdf.set_draw_color(231, 76, 60)
    pdf.set_line_width(0.7)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(4)

    interp = 'SEMNAL POZITIV (PRR >= 2)' if prr_v >= 2 else 'Sub pragul de semnal (PRR < 2)'

    for titlu, continut in [
        ('1. IDENTITATEA SEMNALULUI',
         f'Medicament: {medicament.upper()}\n'
         f'Reactie: {reactie.upper()}\n'
         f'Cazuri detectate: {a_v} | Scor prioritizare: {scor_v:.2f}/10'),
        ('2. METRICI DE DISPROPORTIONALITATE',
         f'PRR = {prr_v:.3f}  =>  {interp}\n'
         f'ROR = {ror_v:.3f}\n'
         f'Scor prioritizare: {scor_v:.2f}/10.00'),
        ('4. RAPOARTE INDIVIDUALE (DRILL-DOWN)', exemple_rapoarte),
        ('5. RECOMANDARI AI SI JUSTIFICARE', recomandari_ai),
    ]:
        pdf.set_font('Arial', 'B', 11)
        pdf.cell(0, 8, ss(titlu), ln=True)
        pdf.set_font('Arial', '', 9)
        pdf.multi_cell(0, 5, ss(continut))
        pdf.ln(2)
        if titlu == '2. METRICI DE DISPROPORTIONALITATE':
            pdf.set_font('Arial', 'B', 11)
            pdf.cell(0, 8, '3. GRAFICE', ln=True)
            if os.path.exists(gpath):
                pdf.image(gpath, x=10, w=185)
            pdf.ln(2)

    pdf.set_y(-18)
    pdf.set_font('Arial', 'I', 7)
    pdf.set_text_color(160, 160, 160)
    pdf.cell(0, 5, 'Document generat automat. Decizia finala apartine expertului in farmacovigilenta.', align='C')

    pdf_path = f'SignalPacket_{safe}.pdf'
    pdf.output(pdf_path)
    if os.path.exists(gpath):
        os.remove(gpath)
    return f'Signal Packet generat cu succes: {pdf_path}'

print('Tool genereaza_pachet_pdf inregistrat.')

Tool genereaza_pachet_pdf inregistrat.


## 4. Construirea Agentului

Folosim **LangGraph** `create_react_agent` cu **Llama 3.3 70B** rulat pe Groq.

### De ce Llama 3.3 70B via Groq?
- **Open-source** — model Meta, fără costuri de licență
- **Gratuit** — Groq oferă 14.400 request-uri/zi pe free tier
- **Rapid** — inferență optimizată pe hardware LPU dedicat
- **Capabil** — 70B parametri, urmează instrucțiuni complexe și apelează tool-uri corect

### Pattern ReAct (Reason + Act)
Agentul ciclează autonom până finalizează toate cele 7 etape:

In [32]:
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

model_ai = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY
)

unelte = [
    filtreaza_dupa_timp,
    normalizeaza_medicament_rxnorm,
    normalizeaza_reactie_meddra,
    calculeaza_metrici_si_trend,
    extrage_cazuri_reprezentative,
    genereaza_pachet_pdf,
]

SYSTEM_PROMPT = (
    'Esti un expert senior in farmacovigilenta (PV). '
    'Misiunea ta este sa investighezi semnale de siguranta medicament-reactie adversa. '
    'Urmeaza OBLIGATORIU acesti pasi in ordine:\n'
    'PAS 1: filtreaza_dupa_timp\n'
    'PAS 2: normalizeaza_medicament_rxnorm\n'
    'PAS 3: normalizeaza_reactie_meddra\n'
    'PAS 4: calculeaza_metrici_si_trend\n'
    'PAS 5: extrage_cazuri_reprezentative\n'
    'PAS 6: formuleaza recomandari bazate pe date (2-3 propozitii clare).\n'
    'PAS 7: genereaza_pachet_pdf\n'
    'Raspunde in limba romana.'
)

agent_farmaco = create_react_agent(
    model=model_ai,
    tools=unelte,
    prompt=SYSTEM_PROMPT
)

print('Agent initializat cu succes:')
print(f'  LLM:        Llama 3.3 70B')
print(f'  Provider:   Groq')
print(f'  Framework:  LangGraph ReAct')
print(f'  Tools:      {len(unelte)}')

Agent initializat cu succes:
  LLM:        Llama 3.3 70B
  Provider:   Groq
  Framework:  LangGraph ReAct
  Tools:      6


## 5. Interfață Gradio

Interfața web permite investigarea semnalelor fără a interacționa cu codul.
`share=True` generează un link public temporar — util pentru demonstrații.

Funcția `ruleaza_investigatie` este bridge-ul dintre UI și agent:
primește inputul din formulare, construiește comanda în limbaj natural,
rulează agentul și returnează raportul text + PDF-ul pentru download.

In [39]:
# ============================================================
# UI — GRADIO
# ============================================================
!pip install gradio -q

import gradio as gr
import shutil

import time

def ruleaza_investigatie(medicament, reactie, an_start, an_stop):
    global _df_curent

    if not medicament.strip() or not reactie.strip():
        return "❌ Completeaza medicamentul si reactia.", None

    # FIX: Gradio returneaza string din Dropdown — convertim explicit la int
    # altfel agentul primeste '2024' (string) si filtreaza pe 0 rapoarte fara eroare
    an_start = int(an_start)
    an_stop  = int(an_stop)

    comanda = (
        f"Investigheaza semnalul de siguranta pentru medicamentul {medicament.upper()} "
        f"si reactia {reactie.upper()}, in perioada {an_start}-{an_stop}. "
        f"Urmeaza toti pasii si genereaza Signal Packet-ul PDF."
    )

    _df_curent = date_unice.copy()

    # KPI 3: masuram timpul real de la cerere la rezultat
    timp_start = time.time()

    for incercare in range(3):
        try:
            rezultat = agent_farmaco.invoke({"messages": [HumanMessage(content=comanda)]})
            raspuns  = rezultat["messages"][-1].content
            break
        except Exception as e:
            eroare = str(e)
            if '503' in eroare or 'UNAVAILABLE' in eroare:
                if incercare < 2:
                    time.sleep(10)
                    continue
            return f"❌ Eroare agent: {eroare}", None
    else:
        return "❌ Server indisponibil. Incearca din nou.", None

    timp_total = time.time() - timp_start

    # KPI 1, 2, 3, 4 — calculate din timpii reali
    timp_manual_min  = 45
    timp_ai_min      = timp_total / 60
    factor_viteza    = timp_manual_min / max(timp_ai_min, 0.01)
    pachete_zi_ai    = (8 * 60) / max(timp_ai_min, 0.01)

    safe     = re.sub(r'[^a-zA-Z0-9]', '_', medicament.upper())[:15]
    pdf_path = f"SignalPacket_{safe}.pdf"
    pdf_exista = os.path.exists(pdf_path)

    kpi_text = (
        f"\n{'='*50}\n"
        f"📊 RAPORT KPI — masurat in timp real\n"
        f"{'='*50}\n"
        f"1. TIMP DE TRIAJ\n"
        f"   Manual:       ~{timp_manual_min} minute\n"
        f"   Cu AI:        {timp_total:.1f} secunde ({timp_ai_min:.2f} minute)\n"
        f"   Factor viteza: {factor_viteza:.0f}x mai rapid\n\n"
        f"2. PACHETE PER ANALIST PER ZI (8h)\n"
        f"   Cu AI:   {pachete_zi_ai:.0f} pachete/zi\n"
        f"   Manual:  ~10 pachete/zi\n\n"
        f"3. TIMP DE LA SEMNAL LA REVIZUIRE\n"
        f"   {timp_total:.1f} secunde de la introducerea datelor\n\n"
        f"4. STANDARDIZARE OUTPUT\n"
        f"   PDF generat: {'✅ DA' if pdf_exista else '❌ NU'}\n"
        f"   Format consistent: ✅ 5 sectiuni identice la fiecare rulare\n"
        f"{'='*50}"
    )

    raspuns_complet = raspuns + kpi_text

    if pdf_exista:
        dest = f"/tmp/{pdf_path}"
        shutil.copy(pdf_path, dest)
        return raspuns_complet, dest
    else:
        return raspuns_complet, None

with gr.Blocks(theme=gr.themes.Soft(), title="PV Signal Triage Copilot") as demo:

    gr.Markdown("""
    # 🩺 Pharmacovigilance Signal Triage Copilot
    Introduceți un medicament și o reacție adversă pentru a genera automat un **Signal Packet** complet.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            medicament_input = gr.Textbox(
                label="💊 Medicament",
                placeholder="ex: DUPIXENT",
                value="DUPIXENT"
            )
            reactie_input = gr.Textbox(
                label="⚠️ Reacție adversă",
                placeholder="ex: DRY EYE",
                value="DRY EYE"
            )
            with gr.Row():
                an_start_input = gr.Dropdown(
                    label="An start",
                    choices=["2020", "2021", "2022", "2023", "2024"],
                    value="2024"
                )
                an_stop_input = gr.Dropdown(
                    label="An stop",
                    choices=["2020", "2021", "2022", "2023", "2024"],
                    value="2024"
                )
            btn = gr.Button("🔍 Analizează Semnalul", variant="primary", size="lg")

        with gr.Column(scale=2):
            output_text = gr.Textbox(
                label="📋 Raport Agent",
                lines=20,
                interactive=False,
                placeholder="Rezultatul investigației va apărea aici..."
            )
            output_pdf = gr.File(
                label="📄 Signal Packet PDF",
                interactive=False
            )

    btn.click(
        fn=ruleaza_investigatie,
        inputs=[medicament_input, reactie_input, an_start_input, an_stop_input],
        outputs=[output_text, output_pdf]
    )

    gr.Examples(
        examples=[
            ["DUPIXENT",  "DRY EYE",      "2024", "2024"],
            ["HUMIRA",    "INFECTION",     "2024", "2024"],
            ["KEYTRUDA",  "PNEUMONITIS",   "2024", "2024"],
            ["OZEMPIC",   "NAUSEA",        "2024", "2024"],
            ["METHOTREXATE", "NAUSEA", "2022", "2024"],
            ["RITUXIMAB", "NAUSEA", "2022", "2024"],
            ["DUPIXENT", "ECZEMA", "2022", "2024"]
        ],
        inputs=[medicament_input, reactie_input, an_start_input, an_stop_input],
        label="Exemple rapide"
    )

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bd8f3feaf427cbca97.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 6. Evaluare și Testare a Tool-urilor

Această secțiune implementează **teste unitare** pentru tool-urile agentului — verifică că fiecare funcție returnează output corect pe date controlate, independent de LLM sau de disponibilitatea API-urilor externe.

Strategia de testare:
- **Date mock** — un DataFrame mic cu valori cunoscute, nu datele reale FAERS
- **Comparație numerică** — PRR/ROR calculate manual vs output tool
- **Teste de robustete** — comportament la date insuficiente, medicamente inexistente, intervale goale
- **Testare end-to-end** — flow complet fara LLM (apeluri directe de functii)


In [ ]:
# ============================================================
# SECTIUNEA 6: TESTE UNITARE SI EVALUARE TOOL-URI
# ============================================================
import traceback

# ── DATE MOCK ──────────────────────────────────────────────
# Cream un DataFrame mic cu valori cunoscute pentru teste deterministe
# Structura: 20 rapoarte, ASPIRIN apare cu NAUSEA de 6 ori (semnal asteptat)
mock_data = {
    'Report_ID': [f'R{i:03d}' for i in range(1, 21)],
    'Date':      ['20240115'] * 6 + ['20240201'] * 6 + ['20240301'] * 8,
    'Sex':       ['M', 'F'] * 10,
    'Drug_Name': (['ASPIRIN'] * 6 + ['IBUPROFEN'] * 6 + ['ASPIRIN'] * 4 + ['PARACETAMOL'] * 4),
    'Role':      ['Suspect'] * 20,
    'Reaction':  (['NAUSEA'] * 4 + ['HEADACHE'] * 2 +
                  ['NAUSEA'] * 3 + ['VOMITING'] * 3 +
                  ['NAUSEA'] * 2 + ['HEADACHE'] * 2 +
                  ['DIZZINESS'] * 2 + ['NAUSEA'] * 2),
    'An_Raport': [2024] * 20
}
df_mock = pd.DataFrame(mock_data)

# Calcul manual asteptat pentru ASPIRIN + NAUSEA:
# a = len(ASPIRIN & NAUSEA) = 4+2 = 6
# b = len(ASPIRIN & !NAUSEA) = 2+2 = 4  (HEADACHE*2 + HEADACHE*2)
# c = len(!ASPIRIN & NAUSEA) = 3+2 = 5  (IBUPROFEN*3 + PARACETAMOL*2)
# d = len(!ASPIRIN & !NAUSEA) = 3+2 = 5  (IBUPROFEN*3 + PARACETAMOL*2)
# PRR = (6/10) / (5/10) = 1.2  => sub prag (< 2)
# ROR = (6*5)/(4*5) = 30/20 = 1.5
a_exp, b_exp, c_exp, d_exp = 6, 4, 5, 5
prr_exp = (a_exp/(a_exp+b_exp)) / (c_exp/(c_exp+d_exp))
ror_exp = (a_exp*d_exp) / (b_exp*c_exp)

print('='*60)
print('TEST SUITE — Pharmacovigilance Signal Triage Copilot')
print('='*60)

rezultate_teste = []

# ── TEST 1: filtreaza_dupa_timp ────────────────────────────
def test_filtrare():
    global _df_curent
    _df_curent = df_mock.copy()
    # Stocam temporar date_unice reale si inlocuim cu mock
    orig = globals().get('date_unice', df_mock)
    import builtins
    # Apelam direct (nu prin agent) pe date mock
    _df_curent = df_mock[df_mock['An_Raport'] == 2024].copy()
    assert len(_df_curent) == 20, f'Asteptat 20, primit {len(_df_curent)}'
    _df_curent_gol = df_mock[df_mock['An_Raport'] == 2019].copy()
    assert len(_df_curent_gol) == 0, 'Interval gol ar trebui sa returneze 0 rapoarte'
    _df_curent = df_mock.copy()  # reseteaza pentru urmatoarele teste
    return 'PASS'

# ── TEST 2: calculeaza_metrici_si_trend — valori numerice ──
def test_metrici_numerice():
    global _df_curent
    _df_curent = df_mock.copy()
    output = calculeaza_metrici_si_trend.invoke({
        'nume_medicament': 'ASPIRIN',
        'reactie_adversa': 'NAUSEA'
    })
    # Verificam ca PRR si ROR sunt in output
    assert 'PRR=' in output or 'PRR_VALUE=' in output, 'PRR lipsa din output'
    assert 'ROR=' in output or 'ROR_VALUE=' in output, 'ROR lipsa din output'
    # Extragem valorile numerice
    import re
    prr_m = re.search(r'PRR[_VALUE]*=([\d\.]+)', output)
    ror_m = re.search(r'ROR[_VALUE]*=([\d\.]+)', output)
    assert prr_m, 'Nu s-a putut extrage PRR numeric'
    assert ror_m, 'Nu s-a putut extrage ROR numeric'
    prr_calc = float(prr_m.group(1))
    ror_calc = float(ror_m.group(1))
    assert abs(prr_calc - prr_exp) < 0.05, f'PRR: asteptat ~{prr_exp:.3f}, primit {prr_calc:.3f}'
    assert abs(ror_calc - ror_exp) < 0.05, f'ROR: asteptat ~{ror_exp:.3f}, primit {ror_calc:.3f}'
    return f'PASS (PRR={prr_calc:.3f} vs exp={prr_exp:.3f}, ROR={ror_calc:.3f} vs exp={ror_exp:.3f})'

# ── TEST 3: cazuri insuficiente (<3) ──────────────────────
def test_cazuri_insuficiente():
    global _df_curent
    _df_curent = df_mock.copy()
    output = calculeaza_metrici_si_trend.invoke({
        'nume_medicament': 'PARACETAMOL',
        'reactie_adversa': 'DIZZINESS'  # 0 cazuri
    })
    assert 'Prea putine cazuri' in output or 'Evans' in output, \
        f'Trebuia refuzat cu mesaj Evans, primit: {output[:80]}'
    return 'PASS'

# ── TEST 4: medicament inexistent ─────────────────────────
def test_medicament_inexistent():
    global _df_curent
    _df_curent = df_mock.copy()
    output = calculeaza_metrici_si_trend.invoke({
        'nume_medicament': 'MEDICAMENT_INEXISTENT_XYZ',
        'reactie_adversa': 'NAUSEA'
    })
    assert 'Prea putine cazuri' in output or 'Evans' in output, \
        f'Trebuia refuzat, primit: {output[:80]}'
    return 'PASS'

# ── TEST 5: drill-down extrage_cazuri_reprezentative ──────
def test_drilldown():
    global _df_curent
    _df_curent = df_mock.copy()
    output = extrage_cazuri_reprezentative.invoke({
        'nume_medicament': 'ASPIRIN',
        'reactie_adversa': 'NAUSEA'
    })
    assert 'cazuri reprezentative' in output.lower() or 'R0' in output, \
        f'Drill-down nu a returnat cazuri: {output[:80]}'
    # Verifica ca sunt max 5 cazuri returnate
    linii_caz = [l for l in output.split('\n') if l.strip().startswith('[')]
    assert len(linii_caz) <= 5, f'Trebuia max 5 cazuri, primit {len(linii_caz)}'
    return f'PASS ({len(linii_caz)} cazuri returnate)'

# ── TEST 6: normalizeaza_reactie_meddra — mapping local ───
def test_meddra_mapping():
    # Sinonim cunoscut trebuie mapt la PT corect
    out1 = normalizeaza_reactie_meddra.invoke({'nume_reactie': 'DIARRHEA'})
    assert 'DIARRHOEA' in out1, f'DIARRHEA trebuia -> DIARRHOEA, primit: {out1}'
    # Termen deja corect
    out2 = normalizeaza_reactie_meddra.invoke({'nume_reactie': 'NAUSEA'})
    assert 'NAUSEA' in out2, f'NAUSEA trebuia pastrat, primit: {out2}'
    # Whitespace extra
    out3 = normalizeaza_reactie_meddra.invoke({'nume_reactie': '  dry eye  '})
    assert 'DRY EYE' in out3, f'dry eye cu spatii nu a fost normalizat: {out3}'
    return 'PASS (DIARRHEA->DIARRHOEA, NAUSEA->NAUSEA, whitespace OK)'

# ── TEST 7: conversie an_start/an_stop la int ──────────────
def test_conversie_ani():
    # Simulam ce face Gradio: trimite string
    an_start_str = '2022'
    an_stop_str  = '2024'
    an_start_int = int(an_start_str)
    an_stop_int  = int(an_stop_str)
    subset = df_mock[
        (df_mock['An_Raport'] >= an_start_int) &
        (df_mock['An_Raport'] <= an_stop_int)
    ]
    assert len(subset) == 20, f'Conversie int() esuata sau filtrare gresita: {len(subset)}'
    # Verifica ca string comparison (bug vechi) ar da 0
    subset_bug = df_mock[
        (df_mock['An_Raport'] >= an_start_str) &
        (df_mock['An_Raport'] <= an_stop_str)
    ]
    # Comportament asteptat: comparatia int>=str fie arunca TypeError, fie da 0
    return f'PASS (int conversion: {len(subset)} rapoarte; fix necesar confirmat)'

# ── RULAM TOATE TESTELE ─────────────────────────────────────
teste = [
    ('T1: filtreaza_dupa_timp',           test_filtrare),
    ('T2: PRR/ROR numeric pe date mock',  test_metrici_numerice),
    ('T3: cazuri insuficiente (Evans)',   test_cazuri_insuficiente),
    ('T4: medicament inexistent',         test_medicament_inexistent),
    ('T5: drill-down max 5 cazuri',       test_drilldown),
    ('T6: MedDRA mapping local',          test_meddra_mapping),
    ('T7: conversie an_start/stop int',   test_conversie_ani),
]

passed = 0
failed = 0
for nume_test, fn_test in teste:
    try:
        detaliu = fn_test()
        print(f'  ✅ {nume_test} — {detaliu}')
        passed += 1
    except Exception as e:
        print(f'  ❌ {nume_test} — EROARE: {e}')
        failed += 1

print('\n' + '='*60)
print(f'REZULTAT FINAL: {passed}/{passed+failed} teste trecute')
if failed == 0:
    print('✅ Toate tool-urile functioneaza corect pe date mock.')
else:
    print(f'⚠️ {failed} test(e) au esuat — verificati output-ul de mai sus.')
print('='*60)


## 7. Concluzii si SGD-uri Impactate

**Pharmacovigilance Signal Triage Copilot** demonstreaza ca un agent AI poate:
- **Reduce timpii de triaj** de la ore/zile la secunde
- **Elimina erorile umane** in calculul PRR/ROR
- **Standardiza** complet pachetele de evidenta
- **Detecta timpuriu** semnale emergente prin trend temporal

Deciziile finale raman la expertii in farmacovigilenta — agentul ii ajuta sa porneasca de la un **top inteligent si contextualizat**.

### SGD-uri impactate
- **SDG 3 (Sanatate si Bunastare)** — detectarea timpurie a riscurilor medicamentoase protejeaza vieti
- **SDG 9 (Inovatie si Infrastructura)** — automatizarea proceselor medicale cu AI
- **SDG 10 (Inegalitati Reduse)** — acces rapid la informatii de siguranta indiferent de resursele echipei PV